# SQL through Kyuubi's JDBC port

`lab.sql()` opens a PyHive connection to Kyuubi on port 10009. Kyuubi starts a Spark SQL
engine for your user the first time (20-40 seconds), then reuses it, so later queries are
quick. Same thing a BI tool or `beeline -u jdbc:hive2://kyuubi:10009/` does.

In [ ]:
import kyuubi_lab as lab          # client/kyuubi_lab.py (on PYTHONPATH in the Jupyter image)
import pandas as pd
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
RID = lab.new_run_id()            # one id for every job in this notebook run
print("Kyuubi REST", lab.REST_URL, "| JDBC", f"{lab.JDBC_HOST}:{lab.JDBC_PORT}", "| user", lab.USER, "| run", RID)

In [ ]:
lab.sql('SHOW DATABASES')

## The Hive table from the core jobs
`py-hive-etl` writes `demo.orders`. Run it if it's missing.

In [ ]:
dbs = set(lab.sql("SHOW DATABASES").iloc[:, 0])
have = "demo" in dbs and "orders" in set(lab.sql("SHOW TABLES IN demo")["tableName"])
if not have:
    lab.run("py-hive-etl", run_id=RID)
lab.sql("SELECT * FROM demo.orders LIMIT 5")

## CTAS into Parquet and ORC

In [ ]:
lab.sql("CREATE DATABASE IF NOT EXISTS nb")
lab.sql("DROP TABLE IF EXISTS nb.region_sales")
lab.sql("""CREATE TABLE nb.region_sales USING parquet AS
           SELECT region, count(*) orders, round(sum(amount), 2) revenue FROM demo.orders GROUP BY region""")
lab.sql("DROP TABLE IF EXISTS nb.product_sales_orc")
lab.sql("""CREATE TABLE nb.product_sales_orc STORED AS ORC AS
           SELECT product, sum(quantity) units FROM demo.orders GROUP BY product""")
lab.sql("SELECT * FROM nb.region_sales ORDER BY revenue DESC")

In [ ]:
df = lab.sql("SELECT * FROM nb.region_sales ORDER BY revenue DESC")
df.plot.bar(x="region", y="revenue", legend=False, title="revenue by region");

## Where did Spark put it?
`DESCRIBE FORMATTED` shows the metastore entry: location on `s3a://warehouse/`, input format.

In [ ]:
d = lab.sql("DESCRIBE FORMATTED nb.product_sales_orc")
d[d.iloc[:, 0].isin(["Location", "InputFormat", "Serde Library", "Provider"])]

## Query plan and session settings

In [ ]:
print(lab.sql("EXPLAIN SELECT region, sum(amount) FROM demo.orders WHERE quantity > 2 GROUP BY region").iloc[0, 0])

In [ ]:
lab.sql("SET spark.sql.shuffle.partitions=8")
lab.sql("SET spark.sql.shuffle.partitions")

## Window functions and a view

In [ ]:
lab.sql("""CREATE OR REPLACE VIEW nb.v_top_product_per_region AS
  SELECT * FROM (
    SELECT region, product, sum(amount) revenue,
           rank() OVER (PARTITION BY region ORDER BY sum(amount) DESC) rnk
    FROM demo.orders GROUP BY region, product) WHERE rnk = 1""")
lab.sql("SELECT * FROM nb.v_top_product_per_region ORDER BY region")

In [ ]:
lab.close_sql()   # the engine stays up in Kyuubi for a while; this just closes our session